# ผู้ช่วยเลือกเมนูจากวัตถุดิบที่มีด้วย RAG

ชื่อผู้จัดทำ: [กรอกชื่อ]  รหัสนักศึกษา: [กรอก]  รายวิชา: [กรอก]

Streamlit URL: [ยังไม่ได้ Deploy — กรอกหลังใช้งานจริง]

GitHub URL: [ยังไม่ได้ Push — กรอกหลังสร้าง repository]

## แนวคิดและระบบ
อ่าน data/recipes.md → แยกสูตรตาม ## และหัวข้อย่อยตาม ### → Cleaning → Structure-based Chunking พร้อมตรวจ token budget → multilingual Sentence Embedding บน CPU → normalized FAISS IndexFlatIP → รวมสูตรเต็มที่เกี่ยวข้อง → ตรวจวัตถุดิบและอุปกรณ์ → context-grounded prompt → Groq API → ตรวจ recipe_id/citations → แสดงรายละเอียดต้นฉบับ

LLM เลือกสูตรจาก Context เป็น JSON แอปแสดงข้อความจากเอกสารโดยตรงเพื่อจำกัด hallucination สูตรตัวอย่าง 20 เมนูสร้างโดย AI สำหรับงานนี้ ต้องตรวจทานก่อนใช้จริง

## การติดตั้งบน Colab เพื่อทดสอบ Retrieval
กรอก GitHub URL ของคุณก่อนรัน ไม่ต้องใส่ API Key เพื่อทดสอบ Retrieval Colab ใช้สำหรับอธิบายและทดสอบโมดูล ไม่ใช่โฮสต์เว็บ Streamlit ถาวร

In [ ]:
REPOSITORY_URL = "YOUR_GITHUB_REPOSITORY_URL"
assert REPOSITORY_URL != "YOUR_GITHUB_REPOSITORY_URL", "กรอก URL GitHub ที่สร้างจริงก่อน"
import subprocess
subprocess.run(["git", "clone", REPOSITORY_URL, "menu-rag"], check=True)


In [ ]:
%cd /content/menu-rag
%pip install -r requirements-dev.txt


In [ ]:
from sentence_transformers import SentenceTransformer
from rag import MODEL
SentenceTransformer(MODEL, device="cpu", cache_folder=".cache/models")


In [ ]:
!python scripts/validate_data.py
!python -m pytest -q
!python scripts/evaluate.py


## การทดสอบ LLM (ทางเลือก)
ไม่บันทึก API Key ลง Notebook ใช้ Colab Secrets ชื่อ GROQ_API_KEY แล้วเรียก Groq โดยตรงตามตัวอย่างนี้ ยังไม่มีผลทดสอบ LLM จริงใน Notebook นี้

In [ ]:
from google.colab import userdata
from groq import Groq
from rag import Retriever, load_recipes, candidates, select_with_llm, render_answer
model = SentenceTransformer(MODEL, device="cpu", cache_folder=".cache/models")
retriever = Retriever(load_recipes("data"), model)
hits = candidates(retriever, "ข้าวผัดไข่ทำอย่างไร", {"ไข่ไก่", "ข้าวสวย", "ต้นหอม"}, set())
client = Groq(api_key=userdata.get("GROQ_API_KEY"), timeout=30, max_retries=0)
selected = select_with_llm(client, "llama-3.3-70b-versatile", "ข้าวผัดไข่ทำอย่างไร", hits)
print(render_answer(selected))


## ผลทดสอบและข้อจำกัด
ดู TEST_REPORT.md และ retrieval_results.json สำหรับผลที่รันจริง ตรวจผล LLM กับ expected_answer ใน test_questions.csv โดยมนุษย์ด้วย ไม่ใช้ LLM-as-Judge เพียงอย่างเดียว

การตรวจของที่มีใช้ชื่อและ alias ที่กำหนด ไม่รับประกันปริมาณเพียงพอ การอ่านภาษาไทยแบบอิสระยังมีข้อจำกัด ไม่มีข้อมูลโภชนาการ/เวลาเป็นตัวเลข จึงปฏิเสธคำถามดังกล่าว ไม่มี Key จะเปิดเว็บได้แต่ไม่สร้างคำตอบปลอม

## ภาพหน้าจอหลังเว็บใช้งานจริง
1. หน้าแรกและข้อมูลคลังสูตร
2. คำตอบเมื่อวัตถุดิบครบ
3. คำตอบที่ขาดเครื่องปรุง
4. เงื่อนไขไมโครเวฟ
5. ถามต่อเมนูที่สอง
6. หลักฐานอ้างอิง
7. ปฏิเสธคำถามโภชนาการ
8. หน้าตั้งค่า Secrets โดยปิดบังคีย์ทั้งหมด

[เพิ่มภาพ Capture จริงภายหลัง — ไม่มีภาพจำลอง]